# Text Preprocessing

## Objective

The goal of this notebook is to clean the customer reviews and convert raw text into a machine-learning-ready format.

The preprocessing steps include:

- Handling missing values
- Lowercasing
- Removing URLs
- Removing HTML tags
- Removing punctuation
- Removing numbers
- Tokenization
- Stopword Removal
- Lemmatization

In [24]:
import pandas as pd
import re
import string
import nltk

In [25]:
# import nltk

nltk.download("punkt")
nltk.download("punkt_tab")      # Required for newer NLTK versions
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [26]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [27]:
df = pd.read_csv("../data/processed/cleaned_reviews.csv")

df.head()

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Review Length,Sentiment
0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates,53,Positive
1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses,303,Positive
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses,500,Neutral
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants,124,Positive
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses,192,Positive


## Remove Unnecessary Columns

The dataset contains an automatically generated index column (`Unnamed: 0`) from the CSV file. It is not useful for training the model and will be removed.

In [28]:
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

df.head()

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Review Length,Sentiment
0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates,53,Positive
1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses,303,Positive
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses,500,Neutral
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants,124,Positive
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses,192,Positive


## Check Missing Values

Before preprocessing, we check whether the text column contains any missing values.

In [29]:
df["Review Text"].isnull().sum()

np.int64(845)

## Remove Missing Reviews

Machine Learning models cannot learn from empty reviews. Therefore, rows with missing review text are removed.

In [30]:
df = df.dropna(subset=["Review Text"])

df.shape

(22641, 12)

## Initialize NLP Tools

Create the stopword list and the lemmatizer object that will be used throughout the preprocessing pipeline.

In [31]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

## Create Text Preprocessing Function

This function performs all text preprocessing steps in a single reusable pipeline.

In [32]:
def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords and apply lemmatization
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)

## Apply Text Preprocessing

Apply the preprocessing function to every customer review.

In [33]:
df.columns

Index(['Clothing ID', 'Age', 'Title', 'Review Text', 'Rating',
       'Recommended IND', 'Positive Feedback Count', 'Division Name',
       'Department Name', 'Class Name', 'Review Length', 'Sentiment'],
      dtype='str')

In [34]:
df["Clean Review"] = df["Review Text"].apply(preprocess_text)

In [35]:
df.to_csv("../data/processed/cleaned_reviews.csv", index=False)

In [36]:
df[["Review Text", "Clean Review"]].head(10)

,Review Text,Clean Review
0,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky sexy comfortable
1,Love this dress! it's sooo pretty. i happene...,love dress sooo pretty happened find store im ...
2,I had such high hopes for this dress and reall...,high hope dress really wanted work initially o...
3,"I love, love, love this jumpsuit. it's fun, fl...",love love love jumpsuit fun flirty fabulous ev...
4,This shirt is very flattering to all due to th...,shirt flattering due adjustable front tie perf...
5,"I love tracy reese dresses, but this one is no...",love tracy reese dress one petite foot tall us...
6,I aded this in my basket at hte last mintue to...,aded basket hte last mintue see would look lik...
7,"I ordered this in carbon for store pick up, an...",ordered carbon store pick ton stuff always try...
8,I love this dress. i usually get an xs but it ...,love dress usually get x run little snug bust ...
9,"I'm 5""5' and 125 lbs. i ordered the s petite t...",im lb ordered petite make sure length wasnt lo...


In [37]:
df[["Review Text", "Clean Review"]].sample(5, random_state=42)

,Review Text,Clean Review
13365,This sweater is so beautiful on. it is thick m...,sweater beautiful thick material make look box...
19834,This piece is almost what i want... i tried on...,piece almost want tried white version x felt l...
18722,Really like this blouse but am returning for a...,really like blouse returning larger size much ...
10635,These are the perfect light weight relaxing su...,perfect light weight relaxing summer pant fabr...
7348,These look nothing like the picture! they are ...,look nothing like picture super highwaisted lo...
